# Model Experiment — Random Forest

Sections:
1. Setup
2. Data Loading
3. Cleaning — MLflow runs per imputation strategy
4. Feature Engineering — engineered features ablation
5. Feature Selection — three selectors compared
6. Training and Hyperparameter Tuning
7. Cross-Validation of the best configuration
8. Final Pipeline + auto-promotion in MLflow Registry

## 1. Setup

In [2]:
import sys, os, warnings, logging, subprocess
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)

if os.path.isdir('/kaggle/working'):
    subprocess.run(['pip', 'install', '-q', 'dagshub', 'mlflow', 'xgboost'], check=True)
    REPO = '/kaggle/working/ML_Asgn2'
    if os.path.isdir(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '--quiet'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/Saba0033/ML_Asgn2.git', REPO], check=True)
    os.chdir(REPO)
    from kaggle_secrets import UserSecretsClient
    os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')

for p in ['.', '..', '/kaggle/working/ML_Asgn2', '/kaggle/working']:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from mlflow.models.signature import infer_signature

from src.data_utils import load_train, split_columns
from src.preprocessing import (
    build_linear_preprocessor, build_tree_preprocessor,
    CorrelationPruner, engineer_features,
)
from src.mlflow_utils import (
    init_tracking, named_run,
    evaluate_classifier, evaluate_train_val,
    cache_architecture_result, register_if_better,
    run_tree_cleaning, run_linear_cleaning,
    run_fe_probe, run_feature_selection, run_hp_tuning,
    compute_cv_summary,
)

RANDOM_STATE = 42
SAMPLE_FRAC = 0.3

In [3]:
init_tracking('RandomForest_Training')

Accessing as Saba0033

Initialized MLflow to track repo "Saba0033/ML_Asgn2"

Repository Saba0033/ML_Asgn2 initialized!

  MLflow experiment: RandomForest_Training
  Tracking URI:      https://dagshub.com/Saba0033/ML_Asgn2.mlflow


## 2. Data Loading

In [4]:
X, y = load_train(sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

engineer = FunctionTransformer(engineer_features, validate=False)

_eng_sample = engineer.transform(X_train.head(200))
num_cols, cat_cols = split_columns(_eng_sample)
print(f'numeric cols:     {len(num_cols)}')
print(f'categorical cols: {len(cat_cols)}')
print(f'train rows: {len(X_train):,}  val rows: {len(X_val):,}')
print(f'fraud rate (train): {y_train.mean():.4f}')

Loading data from: /kaggle/input/competitions/ieee-fraud-detection
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  stratified sample to 177,162 rows (30%)
  memory:  755.6 MB ->  482.4 MB (36.2% reduction)
  final X shape: (177162, 432), fraud rate: 0.0350
numeric cols:     384
categorical cols: 51
train rows: 141,729  val rows: 35,433
fraud rate (train): 0.0350


In [5]:
MODEL_TAG = 'RandomForest'

## 3. Cleaning

Tree models tolerate raw scale and missing values, so cleaning here means choosing an imputation strategy. `run_tree_cleaning` probes two candidates as MLflow runs and returns the better preprocessor:

- `fill=-999` — sentinel imputation, lets the tree split on missingness as a signal.
- `fill=0` — neutral fill; loses the missingness signal but is the natural value for several counters.

In [6]:
preprocessor, best_fill = run_tree_cleaning(
    MODEL_TAG, engineer, num_cols, cat_cols,
    X_train, y_train, X_val, y_val, RANDOM_STATE)

🏃 View run RandomForest_Cleaning_fill-999 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/9de422db584c4ca29f49073b698ebff9
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_Cleaning_fill0 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/7747d26d0ac547b2868e045f6ca7f18a
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
  best fill=0.0  val_auc=0.7882


## 4. Feature Engineering

Two engineered features (defined in `src/preprocessing.engineer_features` so they're shared across architectures and survive a round-trip through the MLflow registry):

- `TransactionAmt_log` — log-scale of the heavy-tailed amount.
- `P_emaildomain_suffix` / `R_emaildomain_suffix` — TLD of payer / recipient email.

In [7]:
fe_pipe = Pipeline([
    ('eng', engineer), ('pre', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
])
fe_metrics = run_fe_probe(MODEL_TAG, fe_pipe, X_train, y_train, X_val, y_val)

🏃 View run RandomForest_FeatureEngineering at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/676982b6cac846eab1197d343788eb09
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
  fe val_auc=0.7882


## 5. Feature Selection

Three selectors compared via `run_feature_selection`. Each is logged as its own MLflow run with `n_features_kept`:

- `VarianceThreshold` — drop near-constant columns.
- `CorrelationPruner` — drop one of every pair with |Pearson r| > 0.95.
- `SelectFromModel(RandomForest)` — embedded importance, `threshold='median'`.

The selector that maximises `selection_score = val_roc_auc - 0.5 * max(0, overfit_gap - 0.02)` wins.

In [8]:
selectors = {
    'variance':    VarianceThreshold(threshold=0.0),
    'correlation': CorrelationPruner(threshold=0.95),
    'model_based': SelectFromModel(
        RandomForestClassifier(n_estimators=80, max_depth=8,
                               n_jobs=-1, random_state=RANDOM_STATE),
        threshold='median'),
}

def build_fs_pipe(sel):
    return Pipeline([('eng', engineer), ('pre', preprocessor), ('sel', sel),
                     ('clf', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])

best_sel, best_sel_metrics, best_sel_kept, best_sel_name = run_feature_selection(
    MODEL_TAG, selectors, build_fs_pipe, X_train, y_train, X_val, y_val)

🏃 View run RandomForest_FeatureSelection_variance at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/e87f31d4123b4ffabe308f65e9dd43b6
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_FeatureSelection_correlation at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/b30eeea730a04d7ba89200309a246a56
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_FeatureSelection_model_based at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/b2a32f2a1a464a3c83dabd9b5df70e6f
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
  best selector: correlation  val_auc=0.8032  kept=290


## 6. Training and Hyperparameter Tuning

We sweep a hand-picked grid that intentionally spans underfit / well-fit / overfit. `run_hp_tuning` logs each combo as its own MLflow run named with its parameter values, then picks the winner by `selection_score` (val AUC penalised when overfit gap > 2%).

In [9]:
hp_grid = [
    {'n_estimators': 50, 'max_depth': 4},
    {'n_estimators': 100, 'max_depth': 8},
    {'n_estimators': 200, 'max_depth': 12},
    {'n_estimators': 300, 'max_depth': 16},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 20},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1},
]

make_estimator = lambda params: RandomForestClassifier(
    **params, n_jobs=-1, random_state=RANDOM_STATE)

def build_pipe(estimator):
    return Pipeline([('eng', engineer), ('pre', preprocessor),
                     ('sel', best_sel), ('clf', estimator)])

best_params, best_train_val = run_hp_tuning(
    MODEL_TAG, hp_grid, make_estimator, build_pipe,
    X_train, y_train, X_val, y_val)

🏃 View run RandomForest_HP_n50_d4 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/456680c9b38145498664fb7ca7b1cfc8
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_HP_n100_d8 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/cb4f5cb9de3b4986a63b90ba2c85469b
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_HP_n200_d12 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/cadaa28e2133434e9b928a5d95595320
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_HP_n300_d16 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/97ed8b2cafeb459a9f22c8956001a0e0
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
🏃 View run RandomForest_HP_n200_dNA_msl20 at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/5a5

## 7. Cross-Validation

3-fold StratifiedKFold on the chosen HP combo. Stratification matters because of the ~3.5% fraud rate. Mean ± std across folds is the headline number we report.

In [10]:
best_pipe = build_pipe(make_estimator(best_params))

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(best_pipe, X_train, y_train, cv=cv,
                           scoring=['roc_auc', 'average_precision'],
                           return_train_score=True, n_jobs=1)
cv_summary = compute_cv_summary(cv_scores)

with named_run(f'{MODEL_TAG}_CrossValidation', tags={'stage':'cross_validation'}):
    mlflow.log_params(best_params)
    mlflow.log_metrics(cv_summary)
for k, v in cv_summary.items():
    print(f'  {k}: {v:.4f}')

🏃 View run RandomForest_CrossValidation at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/599523a3ca744a8ab4040ac456a5b0e1
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
  cv_train_roc_auc_mean: 0.9645
  cv_val_roc_auc_mean: 0.8959
  cv_val_roc_auc_std: 0.0044
  cv_val_pr_auc_mean: 0.5063
  cv_overfit_gap: 0.0687


## 8. Final Pipeline + auto-promotion

1. Refit on `train + val` (val has done its job during HP tuning; deployed model gets all the data).
2. Log the pipeline as an MLflow artifact with `signature` + `input_example`, so it can be loaded later and called on a raw DataFrame.
3. Cache the architecture's headline metrics to `results_cache.json`.
4. Call `register_if_better(...)` — promotes this pipeline to `IEEEFraudBestModel` only if its CV ROC-AUC beats the version currently in the registry. Whichever architecture is best wins automatically.

In [11]:
X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)
best_pipe.fit(X_full, y_full)

final_val = evaluate_classifier(best_pipe, X_val, y_val, prefix='final_val')

signature = infer_signature(X_train.head(5), best_pipe.predict_proba(X_train.head(5)))
with named_run(f'{MODEL_TAG}_FinalPipeline', tags={'stage':'final_pipeline'}) as final_run:
    mlflow.log_params(best_params)
    mlflow.log_metrics(final_val)
    mlflow.log_metrics(cv_summary)
    model_info = mlflow.sklearn.log_model(
        sk_model=best_pipe,
        name='pipeline',
        signature=signature,
        input_example=X_train.head(2),
    )
    mlflow.set_tag('pipeline_uri', model_info.model_uri)
    final_run_id = final_run.info.run_id
    final_model_uri = model_info.model_uri

cache_architecture_result('RandomForest', {
    'best_params':         {k: (v if v is not None else 'None') for k, v in best_params.items()},
    'val_roc_auc':         float(best_train_val['val_roc_auc']),
    'val_pr_auc':          float(best_train_val['val_pr_auc']),
    'overfit_gap':         float(best_train_val['overfit_gap']),
    'cv_val_roc_auc_mean': cv_summary['cv_val_roc_auc_mean'],
    'cv_val_roc_auc_std':  cv_summary['cv_val_roc_auc_std'],
    'cv_val_pr_auc_mean':  cv_summary['cv_val_pr_auc_mean'],
    'best_selector':       best_sel_name,
    'n_features_kept':     int(best_sel_kept),
    'final_run_id':        final_run_id,
})

register_if_better(final_run_id, cv_summary['cv_val_roc_auc_mean'], model_uri=final_model_uri)

2026/05/06 18:45:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_FinalPipeline at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3/runs/99579a6b8aab49dbaa68c9caad6ce93d
🧪 View experiment at: https://dagshub.com/Saba0033/ML_Asgn2.mlflow/#/experiments/3
  registry champion: v3  cv_val_roc_auc_mean=0.9354
  this run: 0.8959  -- not better, skipping


False